In [5]:
import os
from PyPDF2 import PdfReader
import pandas as pd
from openai import OpenAI

##############################################################################
# Before you run this demo, make sure you:
# 1. Have PyPDF2, pandas, and OpenAI import correctly (or use pip)
# 2. replace "sk-123456789" with the true openai api key
# 3. replce "paper X.pdf" with the name of pdfs you want to process, place them in the same folder as this file. 

# You can obtain your own key at https://platform.openai.com/api-keys
my_key = "sk-l23456789"

# List of PDFs to process
pdf_files = ["paper 1.pdf", "paper 2.pdf", "paper 3.pdf"]


# Initialize OpenAI client with API key
client = OpenAI(api_key= my_key)


# Function to convert PDF to plain text
def pdf_to_text(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        return text
    except Exception as e:
        return f"Error reading {file_path}: {str(e)}"

# Function to query OpenAI model
def query_paper(text):
    prompt = f"""
    Please answer the following question truthfully, based on the complete text of the literature provided below.

    Question: Does this paper report experimental findings related to the electrochemical synthesis conditions for aliphatic C-H oxidation reactions?

    Note that to answer "Yes" to this question, all three criteria must be met: (a) wet lab experimental electrochemical synthesis, (b) transformation from C-H to C-OH or C=O, and (c) the reaction occurs on aliphatic carbon. If any of these criteria are not mentioned, the overall answer should be "No.“

    In your response, adhere strictly to the format below and address each part with specific references to the text:
    ***
    Overall Answer: Yes/No
    Part A: Yes/ No
    Reasoning A: A brief justification for this part of your answer, citing relevant sections of the paper
    Part B: Yes/ No
    Reasoning B: A brief justification for this part of your answer, citing relevant sections of the paper
    Part C: Yes/ No
    Reasoning C: A brief justification for this part of your answer, citing relevant sections of the paper
    ***
    Make sure you consider all sentences in the manuscript and always keep the following questions in mind:
    Does the paper involve wet lab experimental procedures in electrochemical synthesis?
    Part A: Does the paper involve experimental procedures and non-computational/non-simulation data in electrochemical synthesis?
    Part B: Does the paper detail C-H oxidation specifically lead to C-H bond (alcohol) or C=O bond (carbonyl) compounds? Note that the coupling products where C-H becomes C-OR group do not count. Also note that only functionalization of C-H bonds without clear indication on C-OH bond or C=O bond does not count. 
    Part C: Is the reaction happening in aliphatic carbon?

    Take a deep breathe and read it, consider all the expectations, analyze and give reasoning in detail step by step. It is very important to my research.

    Literature:
    {text}
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error querying the model: {str(e)}"

# Function to parse the model's response
def parse_response(response):
    try:
        overall = response.split("Overall Answer:")[1].split("\n")[0].strip()
        part_a = response.split("Part A:")[1].split("\n")[0].strip()
        reasoning_a = response.split("Reasoning A:")[1].split("Part B:")[0].strip()
        part_b = response.split("Part B:")[1].split("\n")[0].strip()
        reasoning_b = response.split("Reasoning B:")[1].split("Part C:")[0].strip()
        part_c = response.split("Part C:")[1].split("\n")[0].strip()
        reasoning_c = response.split("Reasoning C:")[1].strip()
        return {
            "Overall": overall,
            "Part A": part_a,
            "Reasoning A": reasoning_a,
            "Part B": part_b,
            "Reasoning B": reasoning_b,
            "Part C": part_c,
            "Reasoning C": reasoning_c,
        }
    except Exception as e:
        return {"Error": f"Error parsing response: {str(e)}"}



# Initialize results
results = []

# Process each PDF
for pdf in pdf_files:
    print(f"Processing {pdf}...")
    pdf_text = pdf_to_text(pdf)
    word_count= len(pdf_text)
    print("___________________")
    #print(f"Text: {pdf_text}")  #to avoid copyright
    print(f"Total words: {word_count}")
    print("___________________")
    print("     ")
    print("     ")
    print("     ")
    print("     ")
    if "Error" in pdf_text:
        response = pdf_text
    else:
        response = query_paper(pdf_text)
    
    # Parse the response
    parsed_response = parse_response(response)
    results.append({"Paper": pdf, **parsed_response})


# Create a DataFrame
df = pd.DataFrame(results)

# Display the DataFrame in the console
print("\nFinal DataFrame:")
print(df)

# Save results to a CSV
df.to_csv("literature_analysis_results.csv", index=False)
print("Results saved to literature_analysis_results.csv")


Processing paper 1.pdf...
___________________
Total words: 17955
___________________
     
     
     
     
Processing paper 2.pdf...
___________________
Total words: 30515
___________________
     
     
     
     
Processing paper 3.pdf...
___________________
Total words: 19299
___________________
     
     
     
     

Final DataFrame:
  Paper        Overall Part A  \
0  paper 1.pdf   No     Yes     
1  paper 2.pdf   No     Yes     
2  paper 3.pdf  Yes     Yes     

  Reasoning A                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [6]:
pd.set_option("display.max_colwidth", None)  # Do not truncate column contents
pd.set_option('display.colheader_justify', 'left')# Align columns to the left

df

,Paper,Overall,Part A,Reasoning A,Part B,Reasoning B,Part C,Reasoning C
0,paper 1.pdf,No,Yes,"The paper describes wet lab experimental conditions for an electrochemical synthesis. The experimental section details the use of an electrochemical setup with a carbon anode and a platinum cathode in acetonitrile, involving the use of electricity for the reaction, confirming the participation of experimental procedures and non-computational data (see section: ""Results and discussion"", paragraph detailing the initial electrosynthetic carbonylation and control experiments).",Yes,"The paper discusses the oxidation of the C(sp3)-H bond at the benzyl position to form carbonyl compounds, specifically aromatic a-diketones, indicating that the C-H to C=O transformation is the intended product of the reaction (see section: ""abstract"" and ""Introduction"").",No,"The reaction described involves the oxidation of a C(sp3)-H bond at the benzyl (aromatic) position rather than aliphatic carbon. The substrate scope focuses on benzyl ketones (aromatic) for carbonylation, thus the reaction occurs on benzylic, not purely aliphatic, carbons (see section: ""Introduction"", and various points throughout noting ""benzyl"" positions).\n```"
1,paper 2.pdf,No,Yes,"The paper details the experimental procedures for electrochemical synthesis, as indicated by their use of a wet lab experimental setup including carbon electrodes, various solvents, and electrolytes, as well as descriptions of the electrochemical cell setup (e.g., ""reticulated vitreous carbon electrodes,"" ""10 mA per mmol of substrate""). They also conducted experiments on a large scale and optimized reaction conditions through experimentation, demonstrating wet lab research.",No,"Although the paper describes the conversion of C-H bonds to form enones and allylic alcohols through oxidation, these transformations do not exclusively lead to the formation of C-OH (alcohols) or C=O (carbonyl) bonds, which is explicitly required by the question criteria. The primary focus is allylic oxidations leading to enone products, which may not directly correspond to C-OH or C=O bond formation.",Yes,"The reaction does occur on aliphatic carbon, as evidenced by the focus on allylic C-H oxidation and the experiments involving substrates like valencene and other cyclohexenes and cyclohexenones, which are composed of aliphatic carbon frameworks.\n***"
2,paper 3.pdf,Yes,Yes,"The paper involves wet lab experimental procedures in electrochemical synthesis. It discusses a practical electrochemical oxidation process, using quinuclidine as a redox mediator and carbon and nickel electrodes, for the chemoselective and scalable oxidation of unactivated C−H bonds (e.g., ""a practical electrochemical oxidation of unactivated C−H bonds"", ""The use of quinuclidine (A) as a mediator allowed the selective electrochemical oxidation"", ""the reaction proceeds at room temperature in a simple undivided cell under constant current conditions"").",Yes,"The paper details the transformation of C-H bonds specifically into C=O (carbonyl) compounds. The electrochemical protocol transforms methylene groups into ketones, as seen in the sclareolide example and other substrates (e.g., ""methylene groups ... could all be selectively transformed into ketones"", ""The mixtures...exhibited a high oxidation potential, rendering the corresponding radical cation sufficiently reactive to engage unactivated C−H bonds."", ""this electrochemical protocol exhibited compatibility with the free hydroxyl group"").",Yes,"The reaction occurs on aliphatic carbon as the paper mentions the oxidation of unactivated methylene and methine moieties, which are aliphatic positions in organic molecules (e.g., ""methylene groups in linear, cyclic, bicyclic, heteroarene-containing, and natural-product-derived substrates"", ""oxidation of unactivated C−H bonds...in an exceedingly simple fashion through an electrochemical process"", ""the oxidation of sclareolide occurred 